In [ ]:
# Import the usual libraries and variables.
%run standard_imports.py

In [ ]:
def period_over_period(current_series, previous_series):
    return pd.Series(((current_series - previous_series) / previous_series) * 100).astype(int)

def install_counts(db, date_range):
    start_date = date_range[0].strftime(MYSQL_DATE_FORMAT)
    end_date = date_range[1].strftime(MYSQL_DATE_FORMAT)
    query = """
        SELECT developer_app.uuid, Count(merchant_app.app_id) AS '{month} Installs'
        FROM merchant_app
        JOIN developer_app ON merchant_app.app_id = developer_app.id
        JOIN developer ON developer_app.developer_id = developer.id
        JOIN account ON account.id = developer.owner_account_id
        WHERE {exclude}
        AND merchant_app.created_time BETWEEN '{start}' AND '{end}'
        GROUP BY merchant_app.app_id
        ORDER BY Count(merchant_app.app_id) DESC
        LIMIT 100
        """.format(month=date_range[0].strftime('%b %Y'), # Dec 2018
                   exclude=NOT_LIKE_DEVELOPER_NAMES_ACCOUNT_EMAILS,
                   start=start_date,
                   end=end_date)
    df = pd.read_sql(query, con=db.conn)
    return df

def uuids_to_names(db, uuids):
    query = """
        SELECT da.uuid, da.name, da.approval_status, d.name
        FROM developer_app da
        JOIN developer d ON da.developer_id=d.id
        WHERE da.uuid IN (""" + str(uuids)[1:-1] + """)
        """
    df = pd.read_sql(query, con=db.conn)
    return df

In [ ]:
db = Db("~/.clover/p801.cfg")
date_ranges = [(datetime.datetime(2018, 1, 1, 0, 0), datetime.datetime(2018, 2, 1, 0, 0)),
               (datetime.datetime(2018, 2, 1, 0, 0), datetime.datetime(2018, 3, 1, 0, 0)),
               (datetime.datetime(2018, 3, 1, 0, 0), datetime.datetime(2018, 4, 1, 0, 0)),
               (datetime.datetime(2018, 4, 1, 0, 0), datetime.datetime(2018, 5, 1, 0, 0)),
               (datetime.datetime(2018, 5, 1, 0, 0), datetime.datetime(2018, 6, 1, 0, 0)),
               (datetime.datetime(2018, 6, 1, 0, 0), datetime.datetime(2018, 7, 1, 0, 0)),
               (datetime.datetime(2018, 7, 1, 0, 0), datetime.datetime(2018, 8, 1, 0, 0)),
               (datetime.datetime(2018, 8, 1, 0, 0), datetime.datetime(2018, 9, 1, 0, 0)),
               (datetime.datetime(2018, 9, 1, 0, 0), datetime.datetime(2018, 10, 1, 0, 0)),
               (datetime.datetime(2018, 10, 1, 0, 0), datetime.datetime(2018, 11, 1, 0, 0)),
               (datetime.datetime(2018, 11, 1, 0, 0), datetime.datetime(2018, 12, 1, 0, 0)),
               (datetime.datetime(2018, 12, 1, 0, 0), datetime.datetime(2019, 1, 1, 0, 0))]
dfs = []
for date_range in date_ranges:
    dfs.append(install_counts(db, date_range))
monthly_install_counts = reduce(lambda x, y: pd.merge(x, y, on='uuid', how='outer'), dfs)
uuids = monthly_install_counts['uuid'].values.tolist()
names = uuids_to_names(db, uuids)
db.close()

In [ ]:
df = pd.merge(names, monthly_install_counts, on='uuid').sort_values(by=['Dec 2018 Installs'], ascending=False)
df[np.isfinite(df['Nov 2018 Installs'])]